# 🐦 Ornith-1.5-35B-A3B on Colab — Ollama + Public API

Run the cells top to bottom. At the end you get a **public HTTPS URL** for the API.

- Model: `ornith-ai/Ornith-1.5-35B-A3B` **Q4_K_M** (~22 GB, 35B MoE / ~3B active)
- Runtime needed: **T4 GPU** (free tier works)
- Final API: OpenAI-compatible at `http://127.0.0.1:11434/v1`, exposed publicly via a Cloudflare quick tunnel

> ⚠️ Keep this browser tab open — free Colab disconnects on idle.
> ⚠️ The tunnel URL changes every time you re-run the tunnel cell (no account needed).

In [ ]:
# 1) Verify the GPU and disk
import shutil
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
g = shutil.disk_usage('/')
print(f"disk: {g.total/1e9:.0f} GB total, {g.free/1e9:.0f} GB free (need ~25 GB)")

In [ ]:
# 2) Install Ollama (prebuilt, with CUDA support) — ~2 min
# zstd fix: the installer needs it on Ubuntu 24.04 images
!apt-get update -qq && apt-get install -y -qq zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
print('install done')

In [ ]:
# 3) Start the Ollama server in the background and verify it is alive
import subprocess, time, urllib.request
!mkdir -p /content/logs
subprocess.run('pkill -f "ollama serve"; true', shell=True)
subprocess.Popen(['bash', '-c', 'OLLAMA_CONTEXT_LENGTH=32768 nohup ollama serve > /content/logs/ollama.log 2>&1 &'],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(20):
    time.sleep(2)
    try:
        v = urllib.request.urlopen('http://127.0.0.1:11434/api/version', timeout=5).read()
        print('ollama alive:', v.decode())
        break
    except Exception:
        pass
else:
    print(open('/content/logs/ollama.log').read()[-1000:])
    raise RuntimeError('ollama did not start')

In [ ]:
# 4) Pull Ornith-1.5-35B-A3B Q4_K_M — ~22 GB, ~5-10 min on Colab bandwidth.
#    It resumes automatically if interrupted: just re-run this cell.
!ollama pull hf.co/ornith-ai/Ornith-1.5-35B-A3B-GGUF:Q4_K_M

In [ ]:
# 5) Load the model (first call loads it onto the GPU, ~4-6 min one-time)
import json, urllib.request
MODEL = 'hf.co/ornith-ai/Ornith-1.5-35B-A3B-GGUF:Q4_K_M'
req = urllib.request.Request(
    'http://127.0.0.1:11434/api/chat',
    data=json.dumps({'model': MODEL,
                     'messages': [{'role': 'user', 'content': 'Reply with exactly: MODEL LOADED'}],
                     'stream': False, 'options': {'num_predict': 8}}).encode(),
    headers={'Content-Type': 'application/json'})
out = json.loads(urllib.request.urlopen(req, timeout=1800).read())
print(out['message']['content'])

In [ ]:
# 6) Expose the API publicly (no account needed — Cloudflare quick tunnel)
import subprocess, time, re
subprocess.run('pkill -f cloudflared; true', shell=True)
!curl -sL -o /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /content/cloudflared
subprocess.Popen(['bash', '-c',
    '/content/cloudflared tunnel --url http://localhost:11434 '
    '--http-host-header="localhost:11434" --no-autoupdate '
    '> /content/logs/tunnel.log 2>&1'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
url = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',
                  open('/content/logs/tunnel.log', errors='ignore').read())
    if m:
        url = m.group(0)
        break
if url:
    print('PUBLIC API URL:', url)
    print('OpenAI endpoint:', url + '/v1')
    print('Copy this URL — it goes into opencode.bat on your PC')
else:
    print(open('/content/logs/tunnel.log', errors='ignore').read()[-800:])
    raise RuntimeError('tunnel did not come up')

In [ ]:
# 7) Verify the public URL works end-to-end (calls back through the internet)
import json, urllib.request
MODEL = 'hf.co/ornith-ai/Ornith-1.5-35B-A3B-GGUF:Q4_K_M'
PUBLIC_URL = url  # from the previous cell
req = urllib.request.Request(
    PUBLIC_URL + '/v1/chat/completions',
    data=json.dumps({'model': MODEL,
                     'messages': [{'role': 'user', 'content': 'Reply with exactly: API IS LIVE'}],
                     'max_tokens': 16}).encode(),
    headers={'Content-Type': 'application/json', 'Authorization': 'Bearer ollama'})
out = json.loads(urllib.request.urlopen(req, timeout=300).read())
print(out['choices'][0]['message']['content'])

## 🔌 Connect OpenCode (on your PC)

**Easiest:** double-click **`setup-opencode_Ornith-1.5-35B-A3B_Q4_K_M_ollama.bat`** in the `colab model` folder on your Desktop,
paste the tunnel URL from cell 6 when asked — done. It writes the provider into
`C:\Users\DELL\.config\opencode\opencode.json` without touching your other settings.

<details><summary>Manual config (what the .bat writes)</summary>

```json
{
  "provider": {
    "colab-ollama": {
      "npm": "@ai-sdk/openai-compatible",
      "name": "Ornith 35B (Colab)",
      "options": {
        "baseURL": "https://YOUR-TUNNEL-URL.trycloudflare.com/v1",
        "apiKey": "ollama"
      },
      "models": {
        "hf.co/ornith-ai/Ornith-1.5-35B-A3B-GGUF:Q4_K_M": {
          "name": "Ornith-1.5-35B-A3B Q4_K_M"
        }
      }
    }
  }
}
```

</details>

Then in OpenCode: pick the `Ornith 35B (Colab)` provider/model and start. Any other tool that
speaks OpenAI chat-completions works too: `base_url = https://<tunnel>/v1`, `api_key = ollama`.

### Notes & limits
- **~12–14 tok/s** on a free T4; prompt processing a few seconds.
- The tunnel + model die when the Colab session stops — re-run cells 3→7 to bring it back
  (the pull in cell 4 resumes instantly since the blob is already on disk).
- New session → new tunnel URL → just re-run `setup-opencode_Ornith-1.5-35B-A3B_Q4_K_M_ollama.bat` with the new URL.
- No authentication: treat the tunnel URL as a secret.